## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [48]:
# import libraries
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import mlflow
import joblib
import requests
import json

In [51]:
df_val = pd.read_csv("C:/Users/miqi/Desktop/med-image-clarity/mlops-and-system-design1/session3/Class Exercise/Churn_Modelling_val.csv")
df_validation = df_val.dropna()
df_validation = df_validation.reset_index(drop=True)
mlflow.set_tracking_uri("http://127.0.0.1:8080")


Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [52]:
def transform(df):
    df = df.copy()
    df[['HasCrCard', 'IsActiveMember', 'Exited']] = df[['HasCrCard', 'IsActiveMember', 'Exited']].astype(int)
    df['Gender'] = df['Gender'].map({'Female': 1, 'Male': 0})
    df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
    return df

df_validation = transform(df_validation)
print(df_validation.head())

   CreditScore Geography  Gender   Age  Tenure    Balance  NumOfProducts  \
0          807     Spain       0  42.0       5       0.00              2   
1          623    France       0  43.0       1       0.00              2   
2          601     Spain       1  44.0       4       0.00              2   
3          506   Germany       0  59.0       8  119152.10              2   
4          560     Spain       1  27.0       7  124995.98              1   

   HasCrCard  IsActiveMember  EstimatedSalary  Exited  
0          1               1         74900.90       0  
1          1               1        146379.30       0  
2          1               0         58561.31       0  
3          1               1        170679.74       0  
4          1               1        114669.79       0  


In [53]:
PATH = "./"
encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
encoder.fit(df_validation[['Geography']])
joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')

['./one_hot_encoder.pkl']

In [54]:
def batch_inference(model_uri, input_df):
    model = mlflow.pyfunc.load_model(model_uri)
    encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')
    df = input_df.copy()
    geo_encoded = encoder.transform(df[['Geography']])
    X = pd.concat([df.drop(columns=['Geography', 'Exited']), geo_encoded], axis=1)
    predictions = model.predict(X)
    return predictions

model_uri = "runs:/ff74216652ec4a73ae3379fa1ad22529/bank_model"
batch_prediction_result = batch_inference(model_uri, df_validation)
print(batch_prediction_result)

[0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 0 1 0 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 1 0 1 1 0
 1 1 0 1 0 1 1 1 0 1 0 1 0 1 1 1 1 1 0 0 0 1 0 1 0 0 1 1 0 1 1 0 1 1 1 1 0
 0 1 1 1 0 0 1 1 1 1 1 0 1 1 1 1 0 0 1 1 1 1 0 0 1 1 0 0 1 1 0 1 0 1 1 1 1
 0 1 0 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 0 1 0 1 0 1 1 0 1 1 1 1 0 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1
 1 0 1 1 0 0 1 1 1 1 0 1 0 0 1 1 0 1 1 1 1 0 1 1 1 0 1 1 1 0 1 1 0 1 1 1 1
 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 0 0 1 1 1 0 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 0
 1 0 0 1 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 1 1 1 0 1 0 1 1 1 1 0 1 0
 1 0 1 1 1 1 0 1 0 0 1 1 1 1 0 1 0 1 1 0 0 1 1 1 1 0 1 1 1 0 1 1 1 0 0 0 1
 1 0 0 0 1 1 0 0 1 0 1 1 0 1 1 0 0 0 0 1 0 1 1 0 1 1 0 0 1 0 1 1 1 0 1 0 0
 0 0 1 1 1 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 0 0 1 1 1 0
 1 0 0 1 1 0 1 1 1 1 1 0 1 1 0 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0
 0 1 1 1 1 1 1 1 1 1 1 1 

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [29]:
# import validation dataset to test inference
df_validation = pd.read_csv("C:/Users/miqi/Desktop/med-image-clarity/mlops-and-system-design1/session3/Class Notes/bank-full_val.csv")

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

##### Batch Inference

In [55]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri, input_df):
    model = mlflow.pyfunc.load_model(model_uri)
    encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')
    df = input_df.copy()
    geo_encoded = encoder.transform(df[['Geography']])
    X = pd.concat([df.drop(columns=['Geography', 'Exited']), geo_encoded], axis=1)
    predictions = model.predict(X)
    return predictions

model_uri = "runs:/ff74216652ec4a73ae3379fa1ad22529/bank_model"
batch_prediction_result = batch_inference(model_uri, df_validation)
print(batch_prediction_result)

[0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 0 1 0 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 1 0 1 1 0
 1 1 0 1 0 1 1 1 0 1 0 1 0 1 1 1 1 1 0 0 0 1 0 1 0 0 1 1 0 1 1 0 1 1 1 1 0
 0 1 1 1 0 0 1 1 1 1 1 0 1 1 1 1 0 0 1 1 1 1 0 0 1 1 0 0 1 1 0 1 0 1 1 1 1
 0 1 0 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 0 1 0 1 0 1 1 0 1 1 1 1 0 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1
 1 0 1 1 0 0 1 1 1 1 0 1 0 0 1 1 0 1 1 1 1 0 1 1 1 0 1 1 1 0 1 1 0 1 1 1 1
 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 0 0 1 1 1 0 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 0
 1 0 0 1 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 1 1 1 0 1 0 1 1 1 1 0 1 0
 1 0 1 1 1 1 0 1 0 0 1 1 1 1 0 1 0 1 1 0 0 1 1 1 1 0 1 1 1 0 1 1 1 0 0 0 1
 1 0 0 0 1 1 0 0 1 0 1 1 0 1 1 0 0 0 0 1 0 1 1 0 1 1 0 0 1 0 1 1 1 0 1 0 0
 0 0 1 1 1 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 0 0 1 1 1 0
 1 0 0 1 1 0 1 1 1 1 1 0 1 1 0 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0
 0 1 1 1 1 1 1 1 1 1 1 1 

In [56]:
# define the model uri that should be used
model_uri = "runs:/ff74216652ec4a73ae3379fa1ad22529/bank_model"

batch_prediction_result = batch_inference(model_uri, df_validation)

print(batch_prediction_result)

[0 0 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 0 1 0 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 1 0 1 1 0
 1 1 0 1 0 1 1 1 0 1 0 1 0 1 1 1 1 1 0 0 0 1 0 1 0 0 1 1 0 1 1 0 1 1 1 1 0
 0 1 1 1 0 0 1 1 1 1 1 0 1 1 1 1 0 0 1 1 1 1 0 0 1 1 0 0 1 1 0 1 0 1 1 1 1
 0 1 0 1 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 0 1 0 1 0 1 1 0 1 1 1 1 0 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0 1 1 1 1 1 0 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1
 1 0 1 1 0 0 1 1 1 1 0 1 0 0 1 1 0 1 1 1 1 0 1 1 1 0 1 1 1 0 1 1 0 1 1 1 1
 1 1 0 1 1 1 1 1 1 1 1 0 1 1 1 0 0 1 1 1 0 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 0
 1 0 0 1 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 1 1 1 0 1 0 1 1 1 1 0 1 0
 1 0 1 1 1 1 0 1 0 0 1 1 1 1 0 1 0 1 1 0 0 1 1 1 1 0 1 1 1 0 1 1 1 0 0 0 1
 1 0 0 0 1 1 0 0 1 0 1 1 0 1 1 0 0 0 0 1 0 1 1 0 1 1 0 0 1 0 1 1 1 0 1 0 0
 0 0 1 1 1 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 0 0 1 1 1 1 1 0 1 1 0 0 0 1 1 1 0
 1 0 0 1 1 0 1 1 1 1 1 0 1 1 0 0 1 1 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 0
 0 1 1 1 1 1 1 1 1 1 1 1 

##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [ ]:
import requests
import json

In [59]:
# import validation dataset to test inference - just one record
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()

df_validation = pd.read_csv("C:/Users/miqi/Desktop/med-image-clarity/mlops-and-system-design1/session3/Class Exercise/Churn_Modelling_val.csv").head(1)

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [ ]:
# transform data - if necessary

In [60]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()

In [63]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(url: str, input: pd.DataFrame):
    encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')
    df = input.copy()
    geo_encoded = encoder.transform(df[['Geography']])
    X = pd.concat([df.drop(columns=['Geography', 'Exited']), geo_encoded], axis=1)
    data = {"dataframe_split": {"columns": list(X.columns), "data": X.values.tolist()}}
    response = requests.post(url, headers={"Content-Type": "application/json"}, data=json.dumps(data))
    return response

response_pandas = online_inference_pandas(url, df_validation)
response_pandas.content

b'{"error_code": "BAD_REQUEST", "message": "Invalid input. Data is not compatible with model signature. Failed to convert column Gender to type \'float64\'. Error: \'ValueError(\\"could not convert string to float: \'Male\'\\")\'", "sqlstate": "KAM00", "error_class": "INVALID_PARAMETER_VALUE"}'

In [64]:
def online_inference_pandas(url: str, input: pd.DataFrame):
    encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')
    df = input.copy()
    geo_encoded = encoder.transform(df[['Geography']])
    X = pd.concat([df.drop(columns=['Geography', 'Exited']), geo_encoded], axis=1)
    data = {"dataframe_split": {"columns": list(X.columns), "data": X.values.tolist()}}
    response = requests.post(url, headers={"Content-Type": "application/json"}, data=json.dumps(data))
    return response

response_pandas = online_inference_pandas(url, df_validation)
response_pandas.content

b'{"error_code": "BAD_REQUEST", "message": "Invalid input. Data is not compatible with model signature. Failed to convert column Gender to type \'float64\'. Error: \'ValueError(\\"could not convert string to float: \'Male\'\\")\'", "sqlstate": "KAM00", "error_class": "INVALID_PARAMETER_VALUE"}'

In [65]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):
    response = requests.post(url, headers={"Content-Type": "application/json"}, data=json.dumps(input))
    return response

    response = requests.post()
    return response

In [68]:
# define the json as required by MLFlow
encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')
df_one = df_validation.copy()
geo_encoded = encoder.transform(df_one[['Geography']])
X = pd.concat([df_one.drop(columns=['Geography', 'Exited']), geo_encoded], axis=1)

input_json = {
    "dataframe_split": {
        "columns": list(X.columns),
        "data": X.values.tolist()
    }
}

In [69]:
response_json = online_inference_json(url, input_json)
response_json.content

b'{"error_code": "BAD_REQUEST", "message": "Invalid input. Data is not compatible with model signature. Failed to convert column Gender to type \'float64\'. Error: \'ValueError(\\"could not convert string to float: \'Male\'\\")\'", "sqlstate": "KAM00", "error_class": "INVALID_PARAMETER_VALUE"}'